### Galaxy multiplets

`GalaxyMultiplets` links galaxies closer than `r_max` in 3D and `r_perp_max` perpendicular to the line of sight, groups linked galaxies into multiplets (pairs, triplets, quadruplets, ...), and cross-correlates each class (singlets included) with the full galaxy sample in $(r_p, \pi)$.

- **Box**: no randoms. Periodic boundaries, fixed line of sight `los`.
- **Cutsky**: pass `randoms_positions`. Cartesian positions with the observer at the origin, midpoint line of sight, Davis-Peebles estimator.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.pyrecon import (
    PyreconBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.multiplets import (
    GalaxyMultiplets,
    identify_multiplets,
)

setup_logging()

#### Box

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)

# Positions centred on 0 are fine: they are wrapped into [0, boxsize) internally
estimator = GalaxyMultiplets(
    backend="pyrecon",  # We need a backend for the geometry, but no mesh is used
    data_positions=data_positions,
    boxsize=boxsize,
    meshsize=32,
)

# Multiplets can be identified without computing the correlation function
multiplets = identify_multiplets(
    estimator.data_positions,
    estimator.backend.boxsize,
    r_max=7.0,
    r_perp_max=1.05,
    los=los,
)
{n: len(pos) for n, pos in multiplets.items()}

In [ ]:
edges = (np.linspace(2.0, 30.0, 14), np.linspace(-60, 60, 121))  # (rp, pi)
result = estimator.compute(r_max=7.0, r_perp_max=1.05, los=los, edges=edges, nthreads=4) # identify_multiplets is called internally
result  # one Count2Correlation per multiplet class

In [ ]:
# Save, then load projected onto wp(rp) and plot
estimator.save(result, "multiplets_box.h5", overwrite=True)
wp = GalaxyMultiplets.load("multiplets_box.h5", project=True)

fig, ax = GalaxyMultiplets.plot(wp, marker="o")
ax.grid(True, alpha=0.3)

In [ ]:
# Access a single class - see lsstypes docs
pair = wp.get(multiplets="pair")
pair.coords("rp"), pair.value()

#### Cutsky

Positions are Cartesian with the observer at the origin, for data and randoms alike (convert RA, DEC, distance upstream). Here we place the box mock at 1000 Mpc/$h$ and cut an angular and radial window.

> Saving cutsky results requires Davis-Peebles support in lsstypes ([adematti/lsstypes#6](https://github.com/adematti/lsstypes/issues/6)). Until then `compute` raises a `NotImplementedError` in cutsky mode; multiplet identification works.

In [ ]:
rng = np.random.default_rng(1)
center = np.array([1000.0, 0.0, 0.0])


def in_window(xyz: np.ndarray) -> np.ndarray:
    """Check if positions are in the cutsky window, defined by RA, Dec, and distance."""
    dist = np.linalg.norm(xyz, axis=1)
    ra = np.rad2deg(np.arctan2(xyz[:, 1], xyz[:, 0]))
    dec = np.rad2deg(np.arcsin(xyz[:, 2] / dist))
    return (np.abs(ra) < 12) & (np.abs(dec) < 10) & (dist > 850) & (dist < 1150)


xyz = data_positions + center
cutsky_positions = xyz[in_window(xyz)]
randoms = rng.uniform(-250, 250, size=(2_000_000, 3)) + center
randoms = randoms[in_window(randoms)][: 5 * len(cutsky_positions)]
len(cutsky_positions), len(randoms)

In [ ]:
estimator = GalaxyMultiplets(
    backend="pyrecon",
    data_positions=cutsky_positions,
    randoms_positions=randoms,  # randoms -> cutsky mode
    boxsize=800.0,  # backend box enclosing the survey
    boxcenter=center,
    meshsize=32,
)
multiplets = identify_multiplets(
    estimator.data_positions,
    boxsize=None,  # cutsky mode
    r_max=7.0,
    r_perp_max=1.05,
    # No LOS needed
)
{n: len(pos) for n, pos in multiplets.items()}

In [ ]:
try:
    result = estimator.compute(r_max=7.0, r_perp_max=1.05, edges=edges, nthreads=4)
    fig, ax = GalaxyMultiplets.plot(result, marker="o")
except NotImplementedError as exc:
    print(exc)